# 00 — Setup and Data Check

Confirms the `casmi` package imports, paths/config resolve correctly, the raw files exist and
match the expected schema, and reports environment versions. No preprocessing happens here --
that's notebook 01.


In [ ]:
import sys
from pathlib import Path

# --- src/ imports (repo-relative; works whether the notebook is launched from repo root or
#     from notebooks/) -------------------------------------------------------------------------
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import casmi
from casmi.paths import get_project_paths
from casmi.config import PreprocessingConfig

## 1. Paths and config


In [ ]:
paths = get_project_paths()
for field, value in paths.as_dict().items():
    print(f"{field:14s} {value}")


In [ ]:
config = PreprocessingConfig()
config


## 2. Raw file presence


In [ ]:
from casmi.io.files import validate_required_files

file_infos = validate_required_files(paths)
for name, info in file_infos.items():
    print(f"{name:20s} exists={info.exists}  size={info.size_bytes / 1e6:.2f} MB")


## 3. Schema validation


In [ ]:
from casmi.data.schema import validate_train_schema, validate_test_schema

train_check = validate_train_schema(paths.train)
test_check = validate_test_schema(paths.test)
print(f"[{'PASS' if train_check.passed else 'FAIL'}] {train_check.name}: {train_check.detail}")
print(f"[{'PASS' if test_check.passed else 'FAIL'}] {test_check.name}: {test_check.detail}")
assert train_check.passed and test_check.passed


## 4. Sample rows


In [ ]:
from casmi.io.parquet import load_columns
from casmi.data.schema import TRAIN_REQUIRED_COLUMNS, TEST_REQUIRED_COLUMNS

_peak_cols = {"ms2_mzs", "ms2_normalized_intensities"}
train_head = load_columns(paths.train, [c for c in TRAIN_REQUIRED_COLUMNS if c not in _peak_cols], max_rows=5)
test_head = load_columns(paths.test, [c for c in TEST_REQUIRED_COLUMNS if c not in _peak_cols], max_rows=5)

print("train (non-peak columns):")
display(train_head)
print("test (non-peak columns):")
display(test_head)


In [ ]:
import pandas as pd

sample_submission = pd.read_csv(paths.sample_submission)
print("sample_submission:", sample_submission.shape)
display(sample_submission.head(3))


## 5. Environment versions


In [ ]:
def _version(modname):
    try:
        return __import__(modname).__version__
    except Exception:
        return "NOT INSTALLED"


print("python      ", sys.version.split()[0])
for mod in ("numpy", "pandas", "pyarrow", "rdkit", "sklearn", "scipy", "joblib"):
    print(f"{mod:12s} {_version(mod)}")


## Summary

If every check above passed, notebook 01 (`01_preprocessing_pipeline.ipynb`) can run
`run_preprocessing(paths, config)` directly.
